# CEG Colab 冷启动论文结果链路

该 Notebook 用于本地没有 GPU 的场景, 在 Google Colab 中从冷启动执行 CEG 论文结果链路。

重要边界:

- Notebook 只负责环境准备和调度 repository modules。
- 正式 records、tables、figures、reports、readiness 和 result package 均由 `scripts/`、`experiments/`、`main/` 中的代码生成。
- Notebook 不手写正式结果表、图表或 claim audit。
- `colab_formal_input_contract.json` 会声明正式运行所需输入文件、字段和第三方脚本接口。
- 若需要真实实验, 请把 `USE_DRY_RUN_INPUTS` 设为 `False`, 并提供事件、阈值、baseline observation 和 metric rows 路径。


In [ ]:
# 1. 配置区
from pathlib import Path

# 若从 Git 仓库冷启动, 填写仓库 URL; 若已经把 CEG 目录上传到 /content/CEG, 可留空。
REPO_URL = ""
# 留空表示使用远端默认分支; 只有需要固定复现实验提交所在分支时才填写。
REPO_BRANCH = ""
REPO_ROOT = Path("/content/CEG")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/CEG")
WORKSPACE_ROOT = DRIVE_OUTPUT_ROOT

# P0 输入冻结使用真实 pilot 工作区。默认不自动执行, 避免在 CSV 未确认前改写正式输入。
PILOT_WORKSPACE_ROOT = DRIVE_OUTPUT_ROOT / "pilot_runs" / "real_pilot_input_workspace_20260617_034500"
RUN_P0_INPUT_FREEZE = False
P0_DRY_RUN = True
P0_REQUIRE_PASS = False

PROFILE = "paper_main_probe"
USE_DRY_RUN_INPUTS = True
REPETITIONS = 1

# 如需在 Colab 上运行外部 baseline 和高级指标命令计划, 设置为 True 并填写下方 root / 输入路径。
RUN_EXTERNAL_PLANS = False

# 正式外部 baseline / metric 计划默认要求 Colab GPU runtime; 若第三方任务明确只需要 CPU, 可改为 False。
REQUIRE_GPU_FOR_EXTERNAL_PLANS = True

# 正式论文实验可开启覆盖率强约束; dry-run 建议保持 False, 因为 dry-run 只验证链路不覆盖完整矩阵。
REQUIRE_EXPERIMENT_COVERAGE = False

# 结果统一落盘到 Google Drive 的 CEG 目录, 并由 helper 按结果类型划分子目录。
# 真实实验模式下使用这些路径。dry-run 模式会自动生成最小输入。
EVENTS_PATH = None
THRESHOLDS_PATH = None
SAMPLE_MANIFEST_PATH = None

# 若没有 thresholds.json, 可从样本清单的 calibration split 自动校准内容阈值。
CALIBRATE_THRESHOLDS = False
THRESHOLD_TARGET_FPR = 0.01
THRESHOLD_CALIBRATION_SPLIT = "calibration"
BASELINE_OBSERVATIONS_PATH = None
METRIC_ROWS_PATH = None

# 若样本清单包含 reference_path 与 watermarked_path, 可先计算轻量 PSNR/SSIM 指标。
COMPUTE_BASIC_IMAGE_METRICS = False

# 外部 baseline / metric 计划。可直接提供 plan, 也可提供 root 让仓库模板物化计划。
BASELINE_PLAN_PATH = None
METRIC_PLAN_PATH = None
BASELINE_ROOT = None
METRIC_ROOT = None
IMAGE_PAIRS_PATH = None
REFERENCE_IMAGE_ROOT = None
GENERATED_IMAGE_ROOT = None
IMAGE_PROMPT_ROWS_PATH = None


In [ ]:
# 2. 冷启动仓库和依赖
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"当前环境未执行 Google Drive mount 或不在 Colab 中: {exc}")

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

if REPO_URL and not REPO_ROOT.exists():
    clone_command = ["git", "clone", "--depth", "1"]
    if REPO_BRANCH:
        clone_command.extend(["--branch", REPO_BRANCH])
    clone_command.extend([REPO_URL, str(REPO_ROOT)])
    subprocess.run(clone_command, check=True)

if not REPO_ROOT.exists():
    raise FileNotFoundError(
        f"未找到仓库目录: {REPO_ROOT}. 请填写 REPO_URL, 或把项目上传到该路径。"
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
sys.path.insert(0, str(REPO_ROOT))
print(f"repo_root={REPO_ROOT}")
print(f"drive_output_root={DRIVE_OUTPUT_ROOT}")
print(f"workspace_root={WORKSPACE_ROOT}")


In [ ]:
# 3. 检查 Colab 环境
from paper_workflow.colab_utils.cold_start import build_colab_environment_summary

environment_summary = build_colab_environment_summary(REPO_ROOT)
environment_summary


In [ ]:
# 4. 写出正式输入模板, 用于准备非 dry-run 实验输入
from paper_workflow.colab_utils.cold_start import write_colab_formal_input_templates

formal_input_templates_manifest = write_colab_formal_input_templates(WORKSPACE_ROOT)
formal_input_templates_manifest


In [ ]:
# 5. 可选: 运行 P0 真实 pilot 输入冻结门禁
from paper_workflow.notebook_utils.protocol_entrypoint import run_p0_input_freeze_from_notebook

if RUN_P0_INPUT_FREEZE:
    p0_input_freeze_report = run_p0_input_freeze_from_notebook(
        PILOT_WORKSPACE_ROOT,
        dry_run=P0_DRY_RUN,
        require_pass=P0_REQUIRE_PASS,
    )
else:
    p0_input_freeze_report = {
        "overall_decision": "skipped",
        "reason": "RUN_P0_INPUT_FREEZE is False",
        "pilot_workspace_root": str(PILOT_WORKSPACE_ROOT),
        "dry_run_default": P0_DRY_RUN,
    }

p0_input_freeze_summary = {
    "overall_decision": p0_input_freeze_report.get("overall_decision"),
    "dry_run": p0_input_freeze_report.get("dry_run", P0_DRY_RUN),
    "first_blocking_gate": (p0_input_freeze_report.get("first_blocking_gate") or {}).get("gate_id"),
    "recommended_next_stage": p0_input_freeze_report.get("recommended_next_stage"),
    "summary": p0_input_freeze_report.get("summary"),
    "pilot_workspace_root": str(PILOT_WORKSPACE_ROOT),
}
p0_input_freeze_summary


In [ ]:
# 6. 生成正式实验运行清单, 用于区分 dry-run 链路和正式论文结果验收
from paper_workflow.notebook_utils.protocol_entrypoint import write_colab_formal_run_checklist_from_notebook

formal_run_checklist = write_colab_formal_run_checklist_from_notebook(
    WORKSPACE_ROOT / "colab_formal_run_checklist.json",
    REPO_ROOT,
    WORKSPACE_ROOT,
    profile=PROFILE,
    use_dry_run_inputs=USE_DRY_RUN_INPUTS,
    run_external_plans=RUN_EXTERNAL_PLANS,
    require_gpu_for_external_plans=REQUIRE_GPU_FOR_EXTERNAL_PLANS,
    require_experiment_coverage=REQUIRE_EXPERIMENT_COVERAGE,
    events_path=EVENTS_PATH,
    thresholds_path=THRESHOLDS_PATH,
    sample_manifest_path=SAMPLE_MANIFEST_PATH,
    compute_basic_image_metrics=COMPUTE_BASIC_IMAGE_METRICS,
    calibrate_thresholds=CALIBRATE_THRESHOLDS,
    threshold_target_fpr=THRESHOLD_TARGET_FPR,
    threshold_calibration_split=THRESHOLD_CALIBRATION_SPLIT,
    baseline_observations_path=BASELINE_OBSERVATIONS_PATH,
    metric_rows_path=METRIC_ROWS_PATH,
    baseline_plan_path=BASELINE_PLAN_PATH,
    metric_plan_path=METRIC_PLAN_PATH,
    baseline_root=BASELINE_ROOT,
    metric_root=METRIC_ROOT,
    image_pairs_path=IMAGE_PAIRS_PATH,
    reference_image_root=REFERENCE_IMAGE_ROOT,
    generated_image_root=GENERATED_IMAGE_ROOT,
    image_prompt_rows_path=IMAGE_PROMPT_ROWS_PATH,
)
formal_run_checklist_summary = {
    "overall_decision": formal_run_checklist["overall_decision"],
    "blocking_issue_count": formal_run_checklist["blocking_issue_count"],
    "formal_input_source_preflight": formal_run_checklist.get("formal_input_source_preflight"),
    "provided_result_file_preflight": formal_run_checklist.get("provided_result_file_preflight"),
    "external_plan_preflight": formal_run_checklist.get("external_plan_preflight"),
    "gpu_readiness": formal_run_checklist.get("gpu_readiness"),
    "acceptance_commands": formal_run_checklist["acceptance_commands"],
}
formal_run_checklist_summary


In [ ]:
# 7. 运行从输入到论文结果包的完整链路
from paper_workflow.notebook_utils.protocol_entrypoint import run_colab_paper_outputs_from_notebook

summary = run_colab_paper_outputs_from_notebook(
    REPO_ROOT,
    WORKSPACE_ROOT,
    profile=PROFILE,
    repetitions=REPETITIONS,
    use_dry_run_inputs=USE_DRY_RUN_INPUTS,
    run_external_plans=RUN_EXTERNAL_PLANS,
    require_gpu_for_external_plans=REQUIRE_GPU_FOR_EXTERNAL_PLANS,
    require_experiment_coverage=REQUIRE_EXPERIMENT_COVERAGE,
    events_path=EVENTS_PATH,
    thresholds_path=THRESHOLDS_PATH,
    sample_manifest_path=SAMPLE_MANIFEST_PATH,
    compute_basic_image_metrics=COMPUTE_BASIC_IMAGE_METRICS,
    calibrate_thresholds=CALIBRATE_THRESHOLDS,
    threshold_target_fpr=THRESHOLD_TARGET_FPR,
    threshold_calibration_split=THRESHOLD_CALIBRATION_SPLIT,
    baseline_observations_path=BASELINE_OBSERVATIONS_PATH,
    metric_rows_path=METRIC_ROWS_PATH,
    baseline_plan_path=BASELINE_PLAN_PATH,
    metric_plan_path=METRIC_PLAN_PATH,
    baseline_root=BASELINE_ROOT,
    metric_root=METRIC_ROOT,
    image_pairs_path=IMAGE_PAIRS_PATH,
    reference_image_root=REFERENCE_IMAGE_ROOT,
    generated_image_root=GENERATED_IMAGE_ROOT,
    image_prompt_rows_path=IMAGE_PROMPT_ROWS_PATH,
)
cold_start_summary_view = {
    "overall_decision": summary["overall_decision"],
    "colab_formal_run_checklist_decision": summary.get("colab_formal_run_checklist_decision"),
    "paper_result_evidence_decision": summary.get("paper_result_evidence_decision"),
    "formal_input_contract_version": (summary.get("colab_formal_input_contract") or {}).get("contract_version"),
    "formal_input_template_count": (summary.get("formal_input_templates_manifest") or {}).get("template_count"),
    "formal_input_source_preflight": summary.get("colab_formal_run_checklist", {}).get("formal_input_source_preflight"),
    "formal_result_gap_decision": (summary.get("colab_formal_result_gap_report") or {}).get("overall_decision"),
    "formal_runbook_path": summary.get("colab_formal_runbook_path"),
    "formal_result_blocking_gap_count": (summary.get("colab_formal_result_gap_report") or {}).get("blocking_gap_count"),
    "colab_paper_result_semantic_check_summary": summary.get("colab_paper_result_semantic_check_summary"),
    "colab_paper_result_semantic_check_failures": summary.get("colab_paper_result_semantic_check_failures"),
    "colab_paper_result_required_group_failures": summary.get("colab_paper_result_required_group_failures"),
    "colab_paper_result_production_trace_summary": summary.get("colab_paper_result_production_trace_summary"),
    "colab_run_bundle_validation_decision": summary.get("colab_run_bundle_validation_decision"),
    "colab_acceptance_decision": summary.get("colab_acceptance_decision"),
    "colab_acceptance_report_decisions": summary.get("colab_acceptance_report_decisions"),
    "colab_bundle_archive_path": summary.get("colab_bundle_archive_path"),
    "colab_bundle_archive_sha256": summary.get("colab_bundle_archive_sha256"),
    "provided_result_files_decision": summary.get("provided_result_files_manifest", {}).get("overall_decision"),
    "provided_result_file_count": summary.get("provided_result_files_manifest", {}).get("copied_file_count"),
    "command_plan": summary["command_plan"],
}
cold_start_summary_view


In [ ]:
# 8. 查看关键输出路径
from pathlib import Path
from paper_workflow.colab_utils.cold_start import build_colab_output_layout

output_layout = build_colab_output_layout(WORKSPACE_ROOT)
paper_outputs_root = Path(output_layout["paper_outputs_root"])
package_root = Path(output_layout["paper_results_package_root"])
colab_bundle_root = Path(output_layout["colab_run_bundle_root"])
provided_results_root = Path(output_layout["provided_results_root"])
archives_root = Path(output_layout["archives_root"])
key_outputs = {
    "pilot_p0_input_freeze_dry_run_report": PILOT_WORKSPACE_ROOT / "pilot_p0_input_freeze_dry_run_report.json",
    "pilot_p0_input_freeze_report": PILOT_WORKSPACE_ROOT / "pilot_p0_input_freeze_report.json",
    "paper_outputs_summary": paper_outputs_root / "paper_outputs_summary.json",
    "paper_readiness_report": paper_outputs_root / "paper_readiness_report.json",
    "paper_results_report": paper_outputs_root / "paper_results_report.md",
    "paper_claim_audit": paper_outputs_root / "artifacts" / "paper_claim_audit.json",
    "colab_output_layout_manifest": WORKSPACE_ROOT / "colab_output_layout_manifest.json",
    "colab_formal_input_contract": WORKSPACE_ROOT / "colab_formal_input_contract.json",
    "formal_input_templates_manifest": WORKSPACE_ROOT / "inputs" / "formal_input_templates_manifest.json",
    "colab_paper_result_index": WORKSPACE_ROOT / "colab_paper_result_index.json",
    "colab_formal_result_gap_report": WORKSPACE_ROOT / "colab_formal_result_gap_report.json",
    "colab_formal_runbook": WORKSPACE_ROOT / "colab_formal_runbook.md",
    "provided_result_files_manifest": provided_results_root / "provided_result_files_manifest.json",
    "result_package_manifest": package_root / "paper_results_package_manifest.json",
    "result_package_validation": package_root / "paper_results_package_validation.json",
    "colab_run_bundle_manifest": colab_bundle_root / "colab_run_bundle_manifest.json",
    "colab_run_bundle_validation": colab_bundle_root / "colab_run_bundle_validation.json",
    "colab_acceptance_report": colab_bundle_root / "colab_acceptance_report.json",
    "acceptance_bundle_validation_cli": colab_bundle_root / "acceptance" / "colab_run_bundle_validation_cli.json",
    "acceptance_evidence_cli": colab_bundle_root / "acceptance" / "paper_result_evidence_cli.json",
    "colab_bundle_archive": archives_root / "ceg_colab_run_bundle.zip",
    "colab_bundle_archive_manifest": archives_root / "colab_bundle_archive_manifest.json",
}
for name, path in key_outputs.items():
    print(f"{name}: {path} exists={path.exists()}")


In [ ]:
# 9. 下载 Colab 运行 bundle, 同时保留到 Drive 类型化 archives 子目录
from pathlib import Path
from paper_workflow.colab_utils.cold_start import create_colab_bundle_archive

archive_manifest = summary.get("colab_bundle_archive_manifest")
if not archive_manifest:
    archive_manifest = create_colab_bundle_archive(
        WORKSPACE_ROOT,
        allow_dry_run=USE_DRY_RUN_INPUTS,
        require_experiment_coverage=REQUIRE_EXPERIMENT_COVERAGE,
        require_external_command_results=RUN_EXTERNAL_PLANS,
    )
archive_path = archive_manifest["archive_path"]
archive_manifest_path = Path(archive_manifest["archive_manifest_path"])
print(archive_path)
print(f"archive_sha256={archive_manifest['archive_sha256']}")
print("expected_archive_name=ceg_colab_run_bundle.zip")
print("offline_acceptance_command=", " ".join(archive_manifest["offline_acceptance_command"]))
print(f"archive_manifest_path={archive_manifest_path}")

try:
    from google.colab import files
    files.download(archive_path)
    files.download(str(archive_manifest_path))
except Exception as exc:
    print(f"当前环境不支持自动下载, 请在文件面板手动下载: {archive_path}")
    print(f"同时建议下载 sidecar manifest: {archive_manifest_path}")
    print(exc)
